# *<center> Sinusoidal quadrupole: the stability diagram </center>*

**Purpose.** This notebook draws the stability diagram of a quadrupole mass filter under the ordinary sinusoidal drive and checks the drawing in flight. It is the baseline half of a pair: the companion notebook `digital_stability.ipynb` draws the same diagram for the two-level (digital) square drive, where the duty cycle becomes a second knob, and the comparison only means something if the sinusoidal map was produced and verified the same way first.

## Background

An ion between quadrupole rods obeys the Mathieu equation. With dimensionless time $\tau = \Omega t/2$, the transverse coordinates satisfy

$$\ddot{u} + \left(a_u - 2 q_u \cos 2\tau\right) u = 0, \qquad
a_x = -a_y = \frac{8eU}{m r_0^2 \Omega^2}, \qquad
q_x = -q_y = \frac{4eV}{m r_0^2 \Omega^2},$$

with $U$ the DC and $V$ the zero-to-peak RF pole voltage and $r_0$ the field radius.

### Why tr M / 2 decides everything

The quantity the charts are built from is less familiar than the charts themselves, so here is where it comes from. The equation above is linear, so the state $(u, \dot u)$ at any time is a linear function of the state one drive period earlier. Collect that linear map into a 2×2 matrix $M$, the **one-period transfer matrix**: advance two independent solutions (unit position and unit velocity) across one period and write the results as columns (Floquet theory calls $M$ the monodromy matrix; in accelerator physics the same object is the one-turn matrix). Because the drive is periodic, flying $n$ periods is applying $M$ $n$ times, so whether an ion survives forever is the question of whether $M^n$ stays bounded — and that is decided entirely by the eigenvalues of $M$.

Two facts pin the eigenvalues down. First, the equation has no velocity term (no damping), so $M$ preserves phase-space area: $\det M = \lambda_1 \lambda_2 = 1$. Second, the characteristic polynomial of a 2×2 matrix involves only $\det M$ and $\mathrm{tr}\,M$, so with the determinant fixed at one,

$$\lambda_{1,2} = \frac{\mathrm{tr}\,M}{2} \pm \sqrt{\left(\frac{\mathrm{tr}\,M}{2}\right)^{2} - 1}.$$

One number, $\mathrm{tr}\,M/2$, now decides the fate:

* $|\mathrm{tr}\,M|/2 \le 1$: the square root is imaginary, the eigenvalues are the complex pair $e^{\pm i \pi \beta}$ on the unit circle with $\cos \pi\beta = \mathrm{tr}\,M/2$, and $M^n$ rotates without growing — the ion oscillates forever, at the secular frequency $f_{sec} = \beta f_{RF}/2$.
* $|\mathrm{tr}\,M|/2 > 1$: the eigenvalues are the real reciprocal pair $e^{\pm \pi \mu}$ with $\cosh \pi\mu = |\mathrm{tr}\,M|/2$, and the $e^{+\pi\mu}$ direction grows without bound — the ion's amplitude multiplies by that factor every period until it meets a rod.

So $|\mathrm{tr}\,M|/2 = 1$ is the stability boundary, the margin maps color by $\log_{10}$ of this one computed number, and the small-$q$ limit recovers the familiar $\beta \approx q/\sqrt{2}$.

The chart in this notebook is drawn from that transfer matrix (the njit `hill_beta_njit`, certified in `notebooks/V02_mathieu_stability.ipynb` against flown secular frequencies), and the flights here are spot checks: a few points of the drawn diagram flown through a real solved rod field, so the map and the instrument are the same physics and not two separate stories.

Nothing below is transcribed from a reference: the charts are computed point by point by `ion_gym.physics.mathieu` when the notebook runs, and the flights use the same solver and integrator as any ion_gym deck. Everything flown uses the 2-D analytic rod quadrupole of `quad_lib.py`: four round rods at the dodecapole-minimising rod-to-field-radius ratio 1.148, solved on a 0.1 mm lattice. One calibration (the RF pseudopotential curvature of the solved field) sets the volts-per-$q$ conversion; after that every $(a,q)$ point on the chart is a voltage the flyer can be handed.

## The instrument, before any statistics

The quad every flight in this notebook uses, rendered from the solver's own electrode mask and the same potential array the kernel integrates in — not from a separate drawing of the shapes. The motion is planar ($x$, $y$); the one view below **is** the principal plane of this 2-D problem.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# ---- session resources: throughput choices, not physics -----------------
# Every flight in this notebook is a single ion, so a worker count would
# change nothing here and none is set. The trajectory-record budget below
# is passed to every build_run, as the workbench passes its own.
from ion_gym.physics.sizing import default_record_budget_gb, resources_line

RECORD_BUDGET_GB = default_record_budget_gb()   # GB the trajectory record
                                   # may use: RECORD_REFUSE_FRACTION of RAM,
                                   # the workbench default. build_run warns
                                   # at half of it and refuses above it.
                                   # None = RAM unreadable, fixed limits.
print(resources_line(None, RECORD_BUDGET_GB))

In [ ]:
# the shared machinery (quad spec, flyer, calibration, band helpers)
# lives beside this notebook in the shipped manuscript_examples folder
import sys
from ion_gym.io.paths import examples_dir, repo_root
STUDY_DIR = repo_root() / "manuscript_examples" / "quadrupole"
if str(STUDY_DIR) not in sys.path:
    sys.path.insert(0, str(STUDY_DIR))

import numpy as np
import matplotlib.pyplot as plt
import time

import quad_lib as QL
from ion_gym.physics.mathieu import (hill_beta_njit,
                                     hill_half_trace_njit)
from ion_gym.progress import quote, track

# one chart style and palette for both stability notebooks
QL.apply_chart_style()
CLR = QL.CHART_COLORS

In [ ]:
# ---- parameters ---------------------------------------------------------
MZ = 100.0            # Th; the one flown mass (the chart is mass-free)
F_RF_MHZ = 1.0        # drive frequency of every flight here
V_CAL = 30.0          # calibration amplitude (q ~ 0.32, mid zone 1)

NA, NQ = 161, 161     # (a, q) grid of the zone-1 chart
A_SPAN = (-0.30, 0.30)
Q_SPAN = (0.0, 1.0)

Q_LADDER = [0.10, 0.25, 0.40, 0.60, 0.80]   # flown zone-1 spot checks
Q_EDGE = (0.90, 0.93)                        # just-inside / just-outside
T_MAX_US = 160.0      # flight length: ~23 secular periods at q = 0.10
DT_NS = 1.0           # certified in V02's dt-sensitivity section

HIGHQ_MAX = 16.0      # how far up q the a = 0 band scan looks
N_BAND_SCAN = 40001   # band-scan resolution: dq = HIGHQ_MAX / (N-1)
N_PHASE = 16          # entry phases per flown transmission estimate

In [ ]:
# render the flown quad from the solver's arrays (display == solver input)
from ion_gym.viz.viz_core import scene_from_simspec, render_mpl

sp_view = QL.quad_spec(V_CAL, f_rf_mhz=F_RF_MHZ, mz=MZ)
scene = scene_from_simspec(
    sp_view, field=None,   # geometry only: the deck has no DC, so a
                           # static-phi overlay would just read 0
    title=f"quad_lib rod quad: r0 = {QL.R0_MM} mm, "
          f"rods {QL.ROD_FACTOR} r0 (solver mask)")
# render_mpl deregisters its figure from pyplot (no double display);
# the returned figure's rich repr IS the display, so it ends the cell
render_mpl(scene)

## 1 · Calibration — one curvature sets every point

$q$ is linear in $V$ at fixed mass and frequency, so one number converts the whole chart to voltages. That number is read from the **solved** field: the RF pseudopotential curvature at the calibration amplitude gives the secular frequency, which inverts to $q$. A textbook $4eV/mr_0^2\Omega^2$ with the nominal $r_0$ would be a second opinion about a different (ideal) quad; the calibration below is about this one.

In [ ]:
# one field solve; quad_lib scales the result to other frequencies as f^2
t0 = time.time()
V_PER_Q, q_cal = QL.calibrate_v_per_q(f_rf_mhz=F_RF_MHZ, v_cal=V_CAL, mz=MZ)
print(f"V_PER_Q = {V_PER_Q:.2f} V per unit q at {F_RF_MHZ} MHz "
      f"(q = {q_cal:.4f} at {V_CAL} V; {time.time()-t0:.0f}s)")

## 2 · The first stability zone in (a, q)

The chart classifies every grid point by both axes at once — the $x$ axis sees $(+a, q)$, the $y$ axis $(-a, q)$, and the filter transmits only where both are stable. As in the digital notebook, the color is the **stability margin**, $\log_{10}$ of the larger of the two axes' $|\mathrm{tr}\,M/2|$: below zero (blue) both axes are stable, and the familiar tongue with its apex near $(a, q) = (0.237, 0.706)$ is the blue region's boundary at zero. The quantity deserves a precise reading, because the colors invite a wrong one. Outside the boundary $|\mathrm{tr}\,M|/2 = \cosh(\pi\mu)$ with $\mu$ the growth exponent, so the red scale is the orbit's amplitude gain per drive period. Inside, $|\mathrm{tr}\,M|/2 = |\cos \pi\beta|$, so the blue deepens where $\beta$ approaches $1/2$ — **not** where the pseudopotential well is deepest. Well depth is a different quantity entirely (it scales with drive amplitude as $q^2$; notebook V03 treats it) and never appears on this map. The black line is not traced from a textbook: it is the $|\mathrm{tr}\,M|/2 = 1$ level of the same computed array the colors show. White is not a third state either: the scale passes through white exactly at $|\mathrm{tr}\,M|/2 = 1$, so the white fringe just outside the line is weak instability — amplitude gain barely above one per period. An ion there is lost eventually but can outlive a finite transit, which is why a real filter transmits partially just outside the boundary and why its resolution improves with residence time. The same margin convention in both notebooks is what makes their diagrams directly comparable. The cost is quoted from a measured per-point time before the grid runs.

In [ ]:
# ---- up-front cost quote (measured on a warm kernel) --------------------
# warm the njit kernel first: a cold probe times the COMPILE, and the
# quoted estimate would be two orders of magnitude high
hill_half_trace_njit(0.1, 0.5)
probe_t0 = time.time()
hill_half_trace_njit(0.1, 0.5)
hill_half_trace_njit(-0.1, 0.5)
per_point_s = time.time() - probe_t0
quote("two-axis transfer-matrix margins (measured per-point)", NA * NQ,
      per_item_s=per_point_s)

aa = np.linspace(*A_SPAN, NA)
qq = np.linspace(*Q_SPAN, NQ)
margin = np.zeros((NA, NQ))
for i, a in track(list(enumerate(aa)), label="zone-1 rows"):
    for j, q in enumerate(qq):
        hx = abs(hill_half_trace_njit(a, q))
        hy = abs(hill_half_trace_njit(-a, q))
        margin[i, j] = np.log10(max(hx, hy, 1e-12))

from matplotlib.colors import TwoSlopeNorm
fig, ax = plt.subplots(figsize=(7.8, 4.8), layout="constrained")
# saturate the reds early so the tongue and its surroundings, not the
# deep-unstable corners, carry the contrast
vmax = np.percentile(margin, 60)
pm = ax.pcolormesh(qq, aa, margin, cmap="RdBu_r",
                   norm=TwoSlopeNorm(vcenter=0.0, vmin=margin.min(),
                                     vmax=vmax), shading="auto",
                   rasterized=True)
fig.colorbar(pm, ax=ax, label="log10 max(|tr M|/2) over x, y "
                              "(< 0 = both stable)")
ax.contour(qq, aa, margin, levels=[0.0], colors="k", linewidths=1.2)
ax.axhline(0, color="k", lw=0.5, alpha=0.6)
ax.plot(Q_LADDER, [0.0] * len(Q_LADDER), "o", color=CLR["flown_ok"],
        markeredgecolor="white", markeredgewidth=1.0, ms=7,
        label="flown in section 3")
ax.plot(Q_EDGE, [0.0, 0.0], "s", color=CLR["zone_mark"],
        markeredgecolor="white", markeredgewidth=1.0, ms=7,
        label="flown edge pair")
ax.grid(False)
ax.set_xlabel("q")
ax.set_ylabel("a")
ax.set_title(f"sinusoidal drive: stability margin ({NA}x{NQ})\n"
             f"black contour: |tr M|/2 = 1 of the same computed array "
             f"(the stability boundary)", fontsize=10)
ax.legend(fontsize=9)
plt.show()

## 3 · Flown spot checks of the drawn diagram

Two checks tie the chart to the instrument. First, the **secular frequency**: at each ladder $q$ the flown ion's smoothed zero-crossing rate must equal $\beta f_{RF}/2$ from the same transfer matrix that coloured the map. Second, the **boundary**: a pair of flights just inside and just outside $q = 0.908$ must survive and be lost respectively. A flight that confirms both says the voltage handed to the flyer and the $q$ on the chart are the same number.

In [ ]:
# ladder flights: secular frequency against the transfer matrix's beta.
# The first flight is timed and prices the rest (measured, not guessed).
def ladder_point(q):
    sp = QL.quad_spec(q * V_PER_Q, waveform="sin", f_rf_mhz=F_RF_MHZ,
                      mz=MZ, t_max=T_MAX_US, dt_ns=DT_NS)
    _m, tr, ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    f_fl = QL.f_zero_cross(tr, ci, sp.geometry.width_mm / 2, F_RF_MHZ)
    return (q, f_fl, hill_beta_njit(0.0, q) / 2 * F_RF_MHZ, st["kind"])

probe_t0 = time.time()
rows = [ladder_point(Q_LADDER[0])]
PER_FLIGHT_S = time.time() - probe_t0
quote(f"remaining ladder flights of {T_MAX_US} us at dt = {DT_NS} ns "
      f"(first flight measured)", len(Q_LADDER) - 1,
      per_item_s=PER_FLIGHT_S)
for q in track(Q_LADDER[1:], label="q ladder"):
    rows.append(ladder_point(q))

print(f"{'q':>6} {'f_flown/MHz':>12} {'beta*f/2':>10} {'rel err':>9}")
for q, f_fl, f_th, kind in rows:
    print(f"{q:6.2f} {f_fl:12.4f} {f_th:10.4f} {abs(f_fl/f_th-1):9.1e}"
          + ("" if kind == 2 else "  LOST (should not happen in zone 1)"))

In [ ]:
# the boundary pair: just inside survives the full flight, just outside
# is lost to a rod; both trajectories are shown as x(t) so the growth
# is visible, not asserted
fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
for ax, q in zip(axes, Q_EDGE):
    sp = QL.quad_spec(q * V_PER_Q, waveform="sin", f_rf_mhz=F_RF_MHZ,
                      mz=MZ, t_max=T_MAX_US, dt_ns=DT_NS)
    _m, tr, ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    ctr = sp.geometry.width_mm / 2
    ax.plot(tr[:, ci["t"]], tr[:, ci["x"]] - ctr, lw=0.6)
    fate = "survives" if st["kind"] == 2 else f"lost at {st['tof']:.1f} us"
    ax.set_ylabel("x (mm)")
    ax.set_title(f"q = {q} ({fate}); boundary at 0.908", fontsize=9)
axes[-1].set_xlabel("t (us)")
plt.tight_layout()
plt.show()

## 4 · The diagram mapped by flight

The strongest check is to re-derive the chart from the instrument: fly one ion at each point of an $(a, q)$ grid spanning the tongue and mark who survives. The DC that sets $a$ comes from the same calibration as the RF that sets $q$ — their ratio is geometry-free ($|a|/q = 2|U|/V$, both voltages acting through the same field curvature), with the sign convention measured in flight (`quad_lib.dc_for_a`). Where the flown markers change from survived to lost must be where the drawn boundary runs; a disagreement would mean the voltages handed to the flyer and the chart's $(a, q)$ are not the same numbers. The launch is 0.1 mm off axis on purpose: near the boundary a formally bounded orbit still swings many times its launch radius, and a 0.5 mm start turns that finite swing into a rod strike — an aperture loss that a stability map must not be blamed for (measured: the near-boundary grid points flip from lost to survived when the launch drops from 0.5 to 0.1 mm).

In [ ]:
# fly the grid: one ion per (a, q) point, survival as the observable.
# The first flight is timed and prices the rest (measured, not guessed).
A_FLOWN = np.linspace(-0.25, 0.25, 7)
Q_FLOWN = np.linspace(0.15, 0.95, 7)

def flown_point(a, q):
    sp = QL.quad_spec(q * V_PER_Q, waveform="sin", f_rf_mhz=F_RF_MHZ,
                      mz=MZ, dc_v=QL.dc_for_a(a, V_PER_Q),
                      x0=0.1, y0=0.07, t_max=80.0, dt_ns=DT_NS)
    _m, _tr, _ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    return st["kind"] == 2

probe_t0 = time.time()
first = flown_point(A_FLOWN[0], Q_FLOWN[0])
per_flight_s = time.time() - probe_t0
quote("map flights of 80 us (first flight measured)",
      len(A_FLOWN) * len(Q_FLOWN) - 1, per_item_s=per_flight_s)
flown = np.zeros((len(A_FLOWN), len(Q_FLOWN)), dtype=bool)
flown[0, 0] = first
for i, a in track(list(enumerate(A_FLOWN)), label="flown map rows"):
    for j, q in enumerate(Q_FLOWN):
        if i == 0 and j == 0:
            continue
        flown[i, j] = flown_point(a, q)

fig, ax = plt.subplots(figsize=(7.8, 4.8), layout="constrained")
ax.contourf(qq, aa, (margin < 0).astype(float), levels=[0.5, 1.5],
            colors=[CLR["x_band"]], alpha=0.18)
ax.contour(qq, aa, margin, levels=[0.0], colors="k", linewidths=1.2)
for i, a in enumerate(A_FLOWN):
    for j, q in enumerate(Q_FLOWN):
        if flown[i, j]:
            ax.plot(q, a, "o", color=CLR["flown_ok"], ms=7.5,
                    markeredgecolor="white", markeredgewidth=1.0)
        else:
            ax.plot(q, a, "x", color=CLR["flown_lost"], ms=7,
                    markeredgewidth=1.8)
ax.grid(False)
ax.plot([], [], "o", color=CLR["flown_ok"], markeredgecolor="white",
        label="flown: survived 80 us")
ax.plot([], [], "x", color=CLR["flown_lost"], label="flown: lost")
ax.set_xlabel("q")
ax.set_ylabel("a")
ax.set_title(f"the tongue re-derived by {len(A_FLOWN)*len(Q_FLOWN)} "
             f"ion_gym flights (shade/contour: transfer-matrix "
             f"boundary)", fontsize=9)
ax.legend(fontsize=9, loc="lower right")
plt.show()

# the chart and the flights must agree except within one grid cell of
# the boundary, where a finite flight cannot decide; report any
# disagreement loudly instead of letting the figure gloss it
n_mismatch = 0
for i, a in enumerate(A_FLOWN):
    for j, q in enumerate(Q_FLOWN):
        predicted = QL.sin_stable(a, q)
        if predicted != flown[i, j]:
            n_mismatch += 1
            print(f"mismatch at (a, q) = ({a:+.3f}, {q:.3f}): "
                  f"chart says {'stable' if predicted else 'unstable'}, "
                  f"flight says {'survived' if flown[i, j] else 'lost'}")
if n_mismatch == 0:
    print(f"all {flown.size} flown points agree with the chart")

## 5 · The a = 0 axis to high q: the sinusoid's higher bands

Zone 1 ends at $q = 0.908$, but the Mathieu equation has further stable bands along $a = 0$. The scan below lists every stable run of the transfer-matrix classification up to $q$ = `HIGHQ_MAX`, with each band's centre, width, and the bandwidth figure $q_c/\Delta q$ — the same number the digital notebook reads as a mass-resolving power $R_{BL}$, since $m \propto 1/q$ at fixed voltage. The widths are the result to look at: they are what make these bands unusable under a sinusoid, and what the digital drive's duty cycle changes.

In [ ]:
# scan the a = 0 axis; each edge is located to one scan step, stated below
# the kernel is warm from section 2, so this probe times the solve
probe_t0 = time.time()
QL.sin_stable(0.0, 1.0)
per_point_s = time.time() - probe_t0
quote("a = 0 band scan (measured per-point)", N_BAND_SCAN,
      per_item_s=per_point_s)
dq_scan = HIGHQ_MAX / (N_BAND_SCAN - 1)
q_axis = np.linspace(0.0, HIGHQ_MAX, N_BAND_SCAN)
margin_q = np.array([np.log10(max(abs(hill_half_trace_njit(0.0, q)),
                                  1e-12)) for q in q_axis])
bands = QL.a0_stable_bands(lambda q: QL.sin_stable(0.0, q),
                           0.0, HIGHQ_MAX, N_BAND_SCAN)
print(f"stable a = 0 bands up to q = {HIGHQ_MAX} "
      f"(edges located to dq = {dq_scan:.1e}):")
print(f"{'q_lo':>9} {'q_hi':>9} {'center':>9} {'width':>9} {'q_c/dq':>9}")
for lo, hi in bands:
    w = hi - lo
    print(f"{lo:9.4f} {hi:9.4f} {(lo+hi)/2:9.4f} {w:9.5f} "
          f"{(lo+hi)/2/w if w > 0 else float('inf'):9.0f}")

# the margin curve is the picture of that table: stable bands are the
# dips below zero, and their narrowness at high q is visible directly
fig, ax = plt.subplots(figsize=(7.8, 3.7), layout="constrained")
ax.plot(q_axis, margin_q, lw=0.9, color=CLR["x_band"])
ax.axhline(0.0, color="k", lw=0.7)
for lo, hi in bands:
    ax.axvspan(lo, hi, color=CLR["flown_ok"], alpha=0.30)
ax.set_xlabel("q")
ax.set_ylabel("log10 |tr M|/2")
ax.set_title("a = 0 stability margin: bands (shaded) are the dips "
             "below zero", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# fly the second band's centre over a ladder of entry phases: formally
# stable, but the intra-period amplitude swing decides who survives
if len(bands) < 2:
    raise ValueError(f"the a = 0 scan found {len(bands)} band(s) up to "
                     f"q = {HIGHQ_MAX}; the second-band flight needs two")
lo2, hi2 = bands[1]
q2 = 0.5 * (lo2 + hi2)
quote(f"entry-phase flights at q = {q2:.3f} (V = {q2*V_PER_Q:.0f} V), "
      f"at the ladder's measured per-flight time (lost ions cost less)",
      N_PHASE, per_item_s=PER_FLIGHT_S)
n_ok = 0
for pf in track(np.linspace(0, 1, N_PHASE, endpoint=False),
                label="entry phases"):
    sp = QL.quad_spec(q2 * V_PER_Q, waveform="sin", phase_frac=pf,
                      f_rf_mhz=F_RF_MHZ, mz=MZ, x0=0.01, y0=0.007,
                      t_max=50.0, dt_ns=DT_NS)
    _m, _tr, _ci, st = QL.fly_one(sp, record_budget_gb=RECORD_BUDGET_GB)
    n_ok += st["kind"] == 2
print(f"second sinusoidal band, centre q = {q2:.3f}: {n_ok}/{N_PHASE} "
      f"entry phases survive 50 us from a 10 um start at "
      f"{q2*V_PER_Q:.0f} V zero-to-peak")

## Summary

* One calibration of the solved field (`V_PER_Q`) converts the whole $(a,q)$ chart to flyable voltages; the chart itself costs milliseconds per point from the transfer matrix.
* The flown secular frequencies on the zone-1 ladder match $\beta f_{RF}/2$ from the same transfer matrix, the edge pair brackets the $q = 0.908$ boundary in flight, and the 49-point flown $(a, q)$ map re-derives the tongue from the instrument itself.
* Along $a = 0$ the sinusoid has higher stable bands, but they are narrow (the table in section 4 carries the widths and edge resolution), the voltage to reach them scales with $q$, and even at a band centre only part of the entry-phase ladder survives.

## Read-out

Every curve here is computed by ion_gym when the notebook runs; nothing is transcribed. The sinusoidal stability diagram is reproduced from the one-period transfer matrix and verified in flight at the stated operating point (m/z `MZ`, `F_RF_MHZ` MHz, the quad_lib rod quad at $r_0$ = 3 mm). The higher $a=0$ bands exist but offer no practical operating point under this drive: their widths put the required voltage stability and the entry-phase sensitivity out of reach. That is the baseline the companion notebook starts from — the digital drive's duty cycle moves and widens exactly these high-$q$ bands, which is what makes a band-limited mass filter there possible. Continue in `digital_stability.ipynb`.